# Pipeline Bronze → Silver — Dataset Marcobre

## Objetivo

Este notebook ejecuta la transformación de datos desde la capa **Bronze** hacia **Silver**, aplicando limpieza, estandarización temporal y cálculo de métricas operacionales necesarias para análisis y reporting.

## Proceso

El flujo incluye:

* Lectura del dataset desde Bronze
* Normalización de datos numéricos y tratamiento de valores nulos
* Generación de claves temporales (`DateKey`, `YearMonth`)
* Cálculo de acumulados mensuales y métricas con ventanas móviles
* Carga incremental en tabla Delta de la capa Silver

## Tabla destino

`dev_silver_curated_minsur.marcobre_fileserver.dataset_master`


In [0]:
%run ./Parametros

In [0]:
TABLE_BRONZE = f"{bronze_env}.marcobre_fileserver.dataset_master_hist"
TABLE_SILVER = f"{silver_env}.marcobre_fileserver.dataset_master_hist"

In [0]:
# Lista de identificadores de widgets
widgets = ['Dia']

# Crear widgets del notebook de Databricks
create_widgets(widgets)
param = get_widget_values(widgets)

In [0]:
dias = int(param["Dia"]) if param["Dia"] else 0

In [0]:
hoy = datetime.today().date()
fecha_corte = hoy - timedelta(days=dias)

datekey_corte = int(fecha_corte.strftime("%Y%m%d"))
datekey_corte

In [0]:
df = spark.read.table(TABLE_BRONZE)

df_clean = df
for field in df.schema.fields:
    if isinstance(field.dataType, (DoubleType, FloatType)):
        df_clean = df_clean.withColumn(
            field.name,
            F.when(F.isnan(F.col(field.name)), None).otherwise(F.col(field.name))
        )

ts = F.to_timestamp("fecha", "yyyy-MM-dd HH:mm:ss")

df_silver = (
    df_clean
    .withColumn("DateKey", F.date_format(ts, "yyyyMMdd").cast("int"))
    .withColumn("YearMonth", F.date_format(ts, "yyyyMM").cast("int"))
)

ColumnasAcum = [
    "consumo_acido_consumo_planta_oxs",
    "chancado_terciario_faja_cv_311"
]

w_acum = (
    Window
    .partitionBy("YearMonth")
    .orderBy("DateKey")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

for col in ColumnasAcum:
    df_silver = df_silver.withColumn(
        f"{col}_acumulado",
        F.sum(F.coalesce(F.col(col), F.lit(0))).over(w_acum)
    )

In [0]:
ColumnasRolling15 = [
    "catodos_cu_cosecha",
    "cu_total_transferido_ew"
]

w_15d = (
    Window
    .orderBy("DateKey")
    .rowsBetween(-14, 0)
)

for col in ColumnasRolling15:
    df_silver = df_silver.withColumn(
        f"{col}_rolling_15d",
        F.sum(F.coalesce(F.col(col), F.lit(0))).over(w_15d)
    )
    
df_silver = df_silver.withColumn(
    "eficiencia_electrodeposicion",
    F.expr("""
        try_divide(
            catodos_cu_cosecha_rolling_15d,
            cu_total_transferido_ew_rolling_15d
        )
    """)
)

In [0]:
df_silver = df_silver.drop(
    "catodos_cu_cosecha_rolling_15d",
    "cu_total_transferido_ew_rolling_15d",
    "YearMonth"
)

cols = ["DateKey"] + [c for c in df_silver.columns if c != "DateKey"]
df_silver = df_silver.select(cols)


In [0]:
df_silver_final = df_silver.filter(F.col("DateKey") >= datekey_corte)

In [0]:
silver_vacia = False
try:
    if spark.table(TABLE_SILVER).limit(1).count() == 0:
        silver_vacia = True
except AnalysisException:
    silver_vacia = True

if silver_vacia:
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_SILVER)
else:
    schema_table = spark.table(TABLE_SILVER).schema
    for field in schema_table:
        if field.name in df_silver_final.columns:
            df_silver_final = df_silver_final.withColumn(
                field.name,
                F.col(field.name).cast(field.dataType)
            )

    spark.sql(f"DELETE FROM {TABLE_SILVER} WHERE DateKey >= {datekey_corte}")
    df_silver_final.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLE_SILVER)